# 실습 5: 센서 590개 진단표 만들기
- 상황: 관리도로 한 개는 봤는데 나머지 589개가 남았다
- 목표: 열 하나를 한 줄로 요약해 쓸 수 없는 열을 걸러낸다

## Step 0. 폴더와 노트북 만들고 데이터 불러오기

In [19]:
import pandas as pd

df = pd.read_csv("../../data/04_secom.csv")

print("행 수, 열 수:", df.shape)


행 수, 열 수: (1567, 592)


## Step 1. 진단 항목 정하기

### 용어 풀이 - 열을 진단할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 진단표 | 열 하나가 한 줄이 되도록 요약한 표. 590개 열이 590줄이 된다 |
| 빈칸 비율 (결측률) | 그 열에서 값이 비어 있는 칸의 비율 |
| 값 종류 수 | 그 열에 서로 다른 값이 몇 가지 들어 있는지 |
| 상수열 | 값 종류가 1개뿐인 열. 처음부터 끝까지 같은 값만 나온다 |
| 스케일 | 값의 크기 단위. 어떤 열은 0~1, 어떤 열은 수천이라 그대로 비교하면 안 된다 |
| 임계값 | 버릴지 말지를 가르는 경계 숫자. 정답이 없어 사람이 정한다 |
| 1차 선별 | 쓸 수 없는 열을 먼저 떨어내는 단계. 쓸모를 따지는 건 그다음이다 |

[내 진단표에 넣을 항목]<br>
1. 빈칸 비율   2. 값 종류 수   3. 표준편차   4. 최소와 최대

In [20]:
# 앞 실습에서 골랐던 센서 이름을 다시 넣는다 (아무 센서나 괜찮습니다)
센서 = "sensor_089"

# isna() — 빈칸이면 참(True), 아니면 거짓(False)
# mean() — 참/거짓의 평균은 곧 참의 비율이 된다 (참=1, 거짓=0이라서)
빈칸비율 = df[센서].isna().mean() * 100

# nunique() — 서로 다른 값이 몇 종류인지 센다
값종류수 = df[센서].nunique()

# std() — 값이 평균에서 얼마나 흩어져 있는지
표준편차 = df[센서].std()

print("열 이름:", 센서)
print("빈칸 비율:", round(빈칸비율, 2), "%")
print("값 종류 수:", 값종류수)
print("표준편차:", round(표준편차, 4))
print("최소~최대:", df[센서].min(), "~", df[센서].max())

열 이름: sensor_089
빈칸 비율: 0.0 %
값 종류 수: 973
표준편차: 53.5373
최소~최대: 1627.4714 ~ 2105.1823


### 문법 노트 - 열 하나를 숫자로 줄이기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| .isna() | 빈칸이면 참, 아니면 거짓 | 빈칸을 세려면 먼저 참/거짓으로 바꿔야 한다 |
| .mean() | 평균 | 참/거짓에 쓰면 참의 비율이 된다 |
| .nunique() | 서로 다른 값의 개수 | 1이면 상수열. 이걸로 바로 판정된다 |
| .std() | 표준편차 | 0에 가까우면 거의 안 변하는 열 |
| .min() .max() | 가장 작은 값 / 가장 큰 값 | 값의 범위. 스케일 감을 잡는다 |

**.isna().mean()이 왜 비율이 되나?**<br>
참을 1, 거짓을 0으로 놓고 평균을 내기 때문이다.<br>
100개 중 4개가 빈칸이면 (1+1+1+1+0+0+...)/100 = 0.04 이므로 4%.<br>
개수를 세고 전체로 나누는 두 단계가 한 줄에 들어간 셈이다.<br>
앞으로 "비율을 구한다" 하면 거의 이 형태가 나온다.

## Step 3. 590개 전체는 AI에게 시키기

In [21]:
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

행목록 = []
for c in sensor_cols:
    빈칸비율 = df[c].isna().mean() * 100
    값종류수 = df[c].nunique()
    표준편차 = df[c].std()
    최솟값 = df[c].min()
    최댓값 = df[c].max()
    행목록.append({
        "열 이름": c,
        "빈칸 비율(%)": round(빈칸비율, 2),
        "값 종류 수": 값종류수,
        "표준편차": 표준편차,
        "최솟값": 최솟값,
        "최댓값": 최댓값,
    })

sensor_diagnosis = pd.DataFrame(행목록)
sensor_diagnosis = sensor_diagnosis.sort_values("빈칸 비율(%)", ascending=False).reset_index(drop=True)

print("표 전체 줄 수:", len(sensor_diagnosis))
sensor_diagnosis.head(10)


표 전체 줄 수: 590


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
0,sensor_293,91.19,92,0.011494,0.0041,0.0831
1,sensor_294,91.19,138,137.692483,82.3233,879.2260
2,sensor_159,91.19,138,406.848810,234.0996,2505.2998
3,sensor_158,91.19,128,0.039538,0.0118,0.2876
4,sensor_493,85.58,226,1.759262,4.8882,21.0443
5,sensor_086,85.58,97,0.002928,0.1053,0.1184
6,sensor_359,85.58,20,0.000395,0.0017,0.0047
7,sensor_221,85.58,69,0.001989,0.0057,0.0240
8,sensor_245,64.96,66,0.084618,0.0003,1.9844
9,sensor_518,64.96,543,4.890663,0.2880,113.2758


### 결과 정리 (실행 결과 기준)

표 전체 줄 수: **590줄** (센서 열 개수만큼)

빈칸 비율 상위 10개:

| 열 이름 | 빈칸 비율(%) | 값 종류 수 | 표준편차 | 최솟값 | 최댓값 |
|---|---|---|---|---|---|
| sensor_293 | 91.19 | 92 | 0.011494 | 0.0041 | 0.0831 |
| sensor_294 | 91.19 | 138 | 137.692483 | 82.3233 | 879.2260 |
| sensor_159 | 91.19 | 138 | 406.848810 | 234.0996 | 2505.2998 |
| sensor_158 | 91.19 | 128 | 0.039538 | 0.0118 | 0.2876 |
| sensor_493 | 85.58 | 226 | 1.759262 | 4.8882 | 21.0443 |
| sensor_086 | 85.58 | 97 | 0.002928 | 0.1053 | 0.1184 |
| sensor_359 | 85.58 | 20 | 0.000395 | 0.0017 | 0.0047 |
| sensor_221 | 85.58 | 69 | 0.001989 | 0.0057 | 0.0240 |
| sensor_245 | 64.96 | 66 | 0.084618 | 0.0003 | 1.9844 |
| sensor_518 | 64.96 | 543 | 4.890663 | 0.2880 | 113.2758 |

빈칸 비율이 가장 높은 4개 열(sensor_293, 294, 159, 158)은 91.19%가 비어 있어, 값이 채워진 행이 전체 1,567행 중 138행뿐이다.

## Step 4. 빈칸 비율 낮은 순 정렬 + 값 종류 1개·표준편차 0인 열 세기

In [21]:
# 빈칸 비율 낮은 순으로 정렬
낮은순 = sensor_diagnosis.sort_values("빈칸 비율(%)", ascending=True).reset_index(drop=True)
print("빈칸 비율 낮은 순 10줄")
display(낮은순.head(10))

# 서로 다른 값이 1개뿐인 열 개수
값종류1개 = (sensor_diagnosis["값 종류 수"] == 1).sum()
print()
print("서로 다른 값의 개수가 1개인 열:", 값종류1개, "개")

# 표준편차가 0인 열 개수
표준편차0 = (sensor_diagnosis["표준편차"] == 0).sum()
print("표준편차가 0인 열:", 표준편차0, "개")


빈칸 비율 낮은 순 10줄


,열 이름,빈칸 비율(%),값 종류 수,표준편차,최솟값,최댓값
0,sensor_121,0.0,1269,0.124304,5.1259,7.5220
1,sensor_521,0.0,1536,5.702366,0.3121,111.7365
2,sensor_522,0.0,9,103.122996,0.0000,1000.0000
3,sensor_523,0.0,1562,7.104435,2.6811,137.9838
4,sensor_524,0.0,1040,4.147581,0.0258,111.3330
5,sensor_525,0.0,1543,20.663414,1.3104,818.0005
6,sensor_527,0.0,1514,0.958428,0.1705,8.2037
7,sensor_572,0.0,811,0.275112,0.9802,2.7395
8,sensor_394,0.0,953,0.038408,0.0342,0.2994
9,sensor_021,0.0,552,0.016737,1.1797,1.4534



서로 다른 값의 개수가 1개인 열: 116 개
표준편차가 0인 열: 116 개


### 결과 정리 (실행 결과 기준 — 위 셀의 실제 출력값)

빈칸 비율 낮은 순(빈칸 0%인 열들) 10개:

| 열 이름 | 빈칸 비율(%) | 값 종류 수 | 표준편차 | 최솟값 | 최댓값 |
|---|---|---|---|---|---|
| sensor_121 | 0.0 | 1269 | 0.124304 | 5.1259 | 7.5220 |
| sensor_521 | 0.0 | 1536 | 5.702366 | 0.3121 | 111.7365 |
| sensor_522 | 0.0 | 9 | 103.122996 | 0.0000 | 1000.0000 |
| sensor_523 | 0.0 | 1562 | 7.104435 | 2.6811 | 137.9838 |
| sensor_524 | 0.0 | 1040 | 4.147581 | 0.0258 | 111.3330 |
| sensor_525 | 0.0 | 1543 | 20.663414 | 1.3104 | 818.0005 |
| sensor_527 | 0.0 | 1514 | 0.958428 | 0.1705 | 8.2037 |
| sensor_572 | 0.0 | 811 | 0.275112 | 0.9802 | 2.7395 |
| sensor_394 | 0.0 | 953 | 0.038408 | 0.0342 | 0.2994 |
| sensor_021 | 0.0 | 552 | 0.016737 | 1.1797 | 1.4534 |

빈칸 비율이 0%인 열이 많아서, 정렬할 때 동점 순서는 `sort_values` 내부 정렬 방식에 따라 달라질 수 있다 — 위 목록은 이 노트북에서 실제 실행했을 때 나온 순서 그대로다.

- **서로 다른 값의 개수가 1개인 열: 116개**
- **표준편차가 0인 열: 116개**

두 조건의 개수가 똑같이 116개로 나왔다 — 값이 하나뿐이면 흩어질 여지가 없어 표준편차도 자연히 0이 되므로, 사실상 같은 열들을 가리키는 결과로 보인다.

## Step 5. 못 쓸 열 걸러내기
빈칸 비율 50% 이상 / 값 종류 1개 / 표준편차 0.001 이하 중 하나라도 해당하면 제외한다. 원본 `sensor_diagnosis`는 그대로 두고, 새 표를 만든다.

In [20]:
# 조건 세 가지 (각각 True/False로 표시)
조건_빈칸 = sensor_diagnosis["빈칸 비율(%)"] >= 50
조건_값종류 = sensor_diagnosis["값 종류 수"] == 1
조건_표준편차 = sensor_diagnosis["표준편차"] <= 0.001

print("빈칸 비율 50% 이상:", 조건_빈칸.sum(), "개")
print("값 종류 1개:", 조건_값종류.sum(), "개")
print("표준편차 0.001 이하:", 조건_표준편차.sum(), "개")

# 셋 중 하나라도 해당하면 제외 대상 (| 는 "또는")
제외조건 = 조건_빈칸 | 조건_값종류 | 조건_표준편차
print("중복 제외 총 제외 대상:", 제외조건.sum(), "개")

# 원본 sensor_diagnosis는 그대로 두고, 제외 대상만 뺀 새 표를 만든다
sensor_diagnosis_필터링 = sensor_diagnosis[~제외조건].reset_index(drop=True)

print()
print("원본 열 개수:", len(sensor_diagnosis))
print("남은 열 개수:", len(sensor_diagnosis_필터링))


빈칸 비율 50% 이상: 28 개
값 종류 1개: 116 개
표준편차 0.001 이하: 127 개
중복 제외 총 제외 대상: 154 개

원본 열 개수: 590
남은 열 개수: 436


### 결과 정리 (실행 결과 기준)

| 조건 | 걸린 열 개수 |
|---|---|
| 빈칸 비율 50% 이상 | 28개 |
| 값 종류 1개 | 116개 |
| 표준편차 0.001 이하 | 127개 |
| **중복 제외 총 제외 대상** | **154개** |

| 표 | 열 개수 |
|---|---|
| 원본 (`sensor_diagnosis`) | 590개 |
| 필터링 후 (`sensor_diagnosis_필터링`) | 436개 |

세 조건을 단순히 더하면 28+116+127=271개지만, 겹치는 열들이 있어서 실제 제외 대상은 154개뿐이다(대부분 "값 종류 1개"와 "표준편차 0.001 이하"가 겹친다). 590개 중 154개를 빼서 436개가 남았다.

## Step 6. 빈칸 비율 기준 비교 (50% vs 30%)
상수열 제외(값 종류 1개)와 표준편차 기준(0.001 이하)은 그대로 두고, 빈칸 비율 기준만 바꿔가며 비교한다. 원본 `sensor_diagnosis`와 앞서 만든 `sensor_diagnosis_필터링`은 건드리지 않는다.

In [19]:
# 상수열 제외, 표준편차 기준은 두 경우 모두 동일하게 적용 (기존과 같은 조건)
조건_값종류 = sensor_diagnosis["값 종류 수"] == 1
조건_표준편차 = sensor_diagnosis["표준편차"] <= 0.001

비교결과 = []
for 기준, 이름 in [(50, "기준 A (50% 이상 제외)"), (30, "기준 B (30% 이상 제외)")]:
    조건_빈칸 = sensor_diagnosis["빈칸 비율(%)"] >= 기준
    제외조건 = 조건_빈칸 | 조건_값종류 | 조건_표준편차

    비교결과.append({
        "기준": 이름,
        "빈칸 과다 제외": 조건_빈칸.sum(),
        "중복 제외 총 제외": 제외조건.sum(),
        "남는 열": (~제외조건).sum(),
    })

비교표 = pd.DataFrame(비교결과)
비교표


,기준,빈칸 과다 제외,중복 제외 총 제외,남는 열
0,기준 A (50% 이상 제외),28,154,436
1,기준 B (30% 이상 제외),32,158,432


### 결과 정리 (실행 결과 기준)

| 기준 | 빈칸 과다 제외 | 중복 제외 총 제외 | 남는 열 |
|---|---|---|---|
| 기준 A (50% 이상 제외) | 28개 | 154개 | 436개 |
| 기준 B (30% 이상 제외) | 32개 | 158개 | 432개 |

기준을 50%에서 30%로 낮추면(더 엄격하게 잡으면) 빈칸 과다로 걸리는 열이 28개→32개로 4개 늘고, 남는 열은 436개→432개로 4개 줄어든다. 상수열·표준편차 조건과 겹치는 부분이 많아서, 기준을 바꿔도 전체 결과에 큰 차이는 나지 않았다.